# EVo: a worked example

EVo models volcanic gas speciation and volume as magma rises from depth toward the surface. Given a magma composition, temperature, fO2, and dissolved volatile contents, EVo tracks how CO2, H2O, SO2, H2S and other species partition between the melt and an exsolving gas phase at each pressure step.

This notebook covers:
1. Setting up the magma composition and run parameters
2. Running EVo
3. Understanding the output columns
4. Using built-in plot functions
5. Building a custom plot

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import evo
import evo.plots as evoplots

## 1. Define the magma composition

The dry (volatile-free) major element composition is provided as a `pd.Series` of oxide weight percents. The example below is basaltic; other example compositions are in `input_files/chem.yaml`.

In [ ]:
composition = pd.Series(
    {
        "SIO2": 47.95,
        "TIO2": 1.67,
        "AL2O3": 17.32,
        "FEO": 10.24,
        "MNO": 0.17,
        "MGO": 5.76,
        "CAO": 10.93,
        "NA2O": 3.45,
        "K2O": 1.99,
        "P2O5": 0.51,
    }
)

## 2. Configure the run

All run settings are passed as a second `pd.Series`. The key options are:

| Parameter | Description |
|---|---|
| `GAS_SYS` | Volatile system to model: `oh`, `coh`, `soh`, `cohs`, `cohsn` |
| `FIND_SATURATION` | If `True`, EVo finds the volatile saturation pressure and decompresses from there |
| `FE_SYSTEM` | If `True`, fO2 is buffered by melt Fe2+/Fe3+ exchange |
| `FO2_buffer` / `FO2_buffer_START` | Rock buffer (`IW`, `FMQ`, `NNO`) and log-unit offset |
| `T_START` | Temperature in Kelvin |

The volatile inputs tell EVo the initial dissolved content of the melt. With `FIND_SATURATION = True`, these values determine the saturation pressure — no need to set `P_START`. See `input_files/env.yaml` for the full list of options and defaults.

In [ ]:
# Set up fixed model parameters

model = {
    "COMPOSITION": "basalt",
    "FIND_SATURATION": True,
    "GAS_SYS": "cohs",
    "FE_SYSTEM": True,
    "FO2_buffer_SET": True,
    "FH2_SET": False,
    "WTH2O_SET": True,
    "WTCO2_SET": True,
    "SULFUR_SET": True,
}

# Then merge with the input parameters for use with EVo.

# Starting conditions
fo2_buffer = "FMQ"  # reference buffer (IW, FMQ or NNO)
dfo2 = 0  # offset from buffer in log units
temp_k = 1473  # temperature, K
h2o_wt_perc = 2.0  # dissolved H2O in the melt, wt%
co2_wt_perc = 0.15  # dissolved CO2 in the melt, wt%
s_ppm = 3000  # dissolved S in the melt, ppm

env = pd.Series(
    model
    | {
        "FO2_buffer": fo2_buffer,
        "FO2_buffer_START": dfo2,
        "T_START": temp_k,
        # Volatile inputs as melt wt fractions (EVo uses wt fraction internally)
        "WTH2O_START": h2o_wt_perc / 100,
        "WTCO2_START": co2_wt_perc / 100,
        "SULFUR_START": s_ppm / 1e6,
    }
)

## 3. Run EVo

`evo.run_evo()` returns a DataFrame for convenient analysis.

In [ ]:
df = evo.run_evo(composition, env)
df

## 4. Understanding the output

The output DataFrame contains one row per pressure step. With `FIND_SATURATION = True` the first row records conditions at the saturation pressure, and subsequent rows follow decompression. The columns fall into several groups:

| Column(s) | Units | Description |
|---|---|---|
| `P` | bar | Pressure |
| `FMQ` (or `NNO` / `IW`) | log units | fO2 relative to buffer |
| `fo2` | bar | Absolute oxygen fugacity |
| `F` | ratio | Molar Fe2O3/FeO in melt |
| `Gas_wt` | wt% | Exsolved gas weight fraction |
| `Exsol_vol%` | vol% | Exsolved gas volume fraction |
| `mH2O`, `mCO2`, `mSO2`, … | mol fraction | Gas phase speciation |
| `wH2O`, `wCO2`, `wSO2`, … | wt fraction | Gas phase speciation |
| `fH2O`, `fCO2`, `fSO2`, … | bar | Species fugacities |
| `mCO2/SO2`, `mH2S/SO2`, … | — | Molar ratios of gas species |
| `H2O_melt`, `CO2_melt`, `S2-_melt`, … | wt% | Volatiles remaining dissolved in melt |
| `Stot_melt` | wt% | Total dissolved S (S2- + S6+) |
| `tot_H`, `tot_C`, `tot_S`, … | ppm | Total elemental budget (use to check conservation) |
| `rho_melt`, `rho_bulk` | kg/m³ | Melt and bulk densities |

In [ ]:
# Pressure, gas fraction, major gas species, and melt residuals
df[
    [
        "P",
        "Gas_wt",
        "Exsol_vol%",
        "mH2O",
        "mCO2",
        "mSO2",
        "mH2S",
        "H2O_melt",
        "CO2_melt",
        "Stot_melt",
    ]
]

## 5. Built-in plots

`evo.plots` provides four functions that each take the output DataFrame and return a matplotlib Figure. All plots use pressure on the y-axis (inverted, so the surface is at the top of the plot).

In [ ]:
# Exsolved gas weight fraction and volume fraction vs pressure
fig = evoplots.plot_gasfraction(df)
plt.tight_layout()
plt.show()

In [ ]:
# Gas phase speciation (mole fractions) vs pressure
fig = evoplots.plot_gasspecies_mol(df)
plt.tight_layout()
plt.show()

In [ ]:
# Dissolved volatile content remaining in the melt vs pressure
fig = evoplots.plot_meltspecies(df)
plt.tight_layout()
plt.show()

In [ ]:
# fO2 evolution: delta-FMQ and absolute log fO2 vs pressure
fig = evoplots.plot_fo2FMQ(df)
plt.tight_layout()
plt.show()

## 6. Custom analysis

The DataFrame can be used directly for any custom analysis. Here we plot H2S/SO2 to show how it evolves during decompression alongside the fO2 path.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

# H2S/SO2 molar ratio — a proxy measurable at volcanic vents
axes[0].plot(df["mH2S/SO2"], df["P"])
axes[0].set_xlabel("H$_2$S / SO$_2$ (molar)")
axes[0].set_ylabel("Pressure (bar)")
axes[0].set_xscale("log")
axes[0].set_title("Gas ratio")

# fO2 relative to FMQ
axes[1].plot(df["FMQ"], df["P"])
axes[1].set_xlabel(r"$\Delta$FMQ")
axes[1].set_title("fO2 path")

# invert the axis
axes[0].invert_yaxis()
plt.suptitle("Basalt COHS degassing at FMQ+0")
plt.tight_layout()
plt.show()